# Beginner LangChain Chatbot with Memory

In this lesson we will build a small chatbot that remembers the conversation in one session.

We will use:
- `ChatOpenAI` to call a chat model
- a prompt with a history placeholder
- `InMemoryChatMessageHistory` to store messages while this notebook is running
- `RunnableWithMessageHistory` to connect memory to the chain

This is application memory: it is stored in Python memory and disappears when the notebook kernel stops.

## 1. Configure your API key

Before running the next cell, set `OPENAI_API_KEY` in your environment or create a local `.env` file. Never commit the key to Git.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

print("API key found.")

API key found.


## 2. Make a first chatbot response

A LangChain chat model takes messages as input and returns an AI message.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

response = model.invoke([
    HumanMessage(content="Hello! I am learning LangChain.")
])

print(response.content)

That's great to hear! LangChain is a powerful framework for building applications with language models. It provides tools for managing prompts, chaining together different components, and integrating with various data sources. What specific aspects of LangChain are you interested in learning about? Are you looking for tutorials, examples, or help with a specific project?


## 3. Add a prompt and conversation history

The history is kept separately for each `session_id`. This lets one application support multiple conversations.

In [4]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly tutor for beginners learning LangChain."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{messages}"),
])

chain = prompt | model
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="history",
)

d:\Repositories\Langchain-Teaching\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 4. Chat using memory

Run the next two cells. The second question can use information from the first because both calls use the same session ID.

In [5]:
config = {"configurable": {"session_id": "beginner-demo"}}

first_reply = chatbot.invoke(
    {"messages": "My name is Alex and I am learning Python."},
    config=config,
)

print(first_reply.content)

Hi Alex! That's great to hear! Python is a fantastic language to learn, especially if you're interested in fields like data science, web development, or automation. How can I assist you with your Python learning journey? Do you have any specific questions or topics you're curious about?


In [6]:
second_reply = chatbot.invoke(
    {"messages": "What is my name, and what am I learning?"},
    config=config,
)

print(second_reply.content)

print(f"Messages in this session: {len(store['beginner-demo'].messages)}")

Your name is Alex, and you are learning Python. If you have any questions or need help with Python concepts, feel free to ask!
Messages in this session: 4


## 5. Try an exercise

1. Change the system message so the chatbot teaches a different subject.
2. Use a new session ID and ask the same question. What changes?
3. Inspect `store['beginner-demo'].messages` to see the saved conversation.
4. Restart the kernel and explain why the memory is gone.

For production applications, replace the in-memory store with a persistent history database and add authentication, rate limits, and error handling.